# OTDR + TDGSA: Undersea Cable Integrity Monitoring

Optical time-domain reflectometry extended with dispersive phase retrieval.
Ship-borne or buoy-mounted. Detects cuts, taps, bends, and strain in real time.

In [ ]:
from sympy import *
from IPython.display import display, Markdown

init_printing(use_latex='mathjax')

def step(label, expr=None):
    display(Markdown(f'**{label}**'))
    if expr is not None:
        display(expr)

def section(title):
    display(Markdown(f'---\n## {title}'))

## §1 — OTDR: Pulse In, Reflections Out

In [ ]:
section('OTDR Principle')

display(Markdown(r'''
**Optical Time-Domain Reflectometry** (OTDR):
Launch a pulse into a fiber. Measure backscattered and reflected light vs time.
Time of flight → distance to event.

```
Pulse source
     │
     ▼
  ───────────────────────────────── fiber ─────── X (fault at distance d)
     ▲
     │ backscatter + reflection
  Detector
```

**Events detected**:
| Event | Signature |
|---|---|
| Rayleigh backscatter | continuous exponential decay (normal) |
| Connector / splice | step down in power |
| Fresnel reflection (cleave/break) | sharp spike up then drop |
| Fiber end | reflection then noise floor |
| Tap / evanescent coupler | power loss without reflection |
| Bend / strain | increased loss, shift in backscatter slope |
'''))

c_fiber, n_eff, t_tof = symbols('c n t', positive=True)

# Distance from time of flight
d = c_fiber * t_tof / (2 * n_eff)
step('Distance to event: d = c·t/(2n_eff) =', d)
step('Factor of 2: pulse travels out AND back')

# Numeric
c_val = 3e8
n_val = 1.468  # SMF-28 at 1550nm
print(f'SMF-28 (n=1.468): 1 ns round-trip = {c_val*1e-9/(2*n_val)*100:.2f} cm')
print(f'1 μs round-trip  = {c_val*1e-6/(2*n_val)/1000:.3f} km')
print(f'10 ms round-trip = {c_val*10e-3/(2*n_val)/1000:.0f} km  (transpacific)')

## §2 — Rayleigh Backscatter: The Normal Baseline

In [ ]:
section('Rayleigh Backscatter')

alpha, z, P0 = symbols('alpha z P_0', positive=True)

# Power vs distance (one-way attenuation, round trip)
P_back = P0 * exp(-2*alpha*z)  # factor 2: out and back
step('Backscattered power P(z) = P₀·e^(−2αz) =', P_back)

# dB scale
P_dB = 10*log(P_back/P0, 10)
step('In dB: P(z)/P₀ [dB] =', simplify(P_dB))
step('= −2α·z·10/ln(10) = −(2α/0.2303)·z  →  linear on dB plot')

# Attenuation coefficient
display(Markdown(r'''
**SMF-28 at 1550 nm**: $\alpha = 0.2$ dB/km = $0.046$ km$^{-1}$

A straight line on the OTDR dB plot = healthy fiber.
Any deviation = event worth investigating.

**Slope change** (strain or temperature):
$$\Delta\alpha = \frac{d\alpha}{d\epsilon}\cdot\Delta\epsilon + \frac{d\alpha}{dT}\cdot\Delta T$$
Distributed sensing: measure $\alpha(z)$ continuously → full strain/temp map.
'''))

# Dead zone
tau_pulse, v_g = symbols('tau_p v_g', positive=True)
dead_zone = v_g * tau_pulse / 2
step('Dead zone (cannot resolve events closer than): Δz_min = v_g·τ_pulse/2 =', dead_zone)
print(f'1 ns pulse → dead zone = {3e8/1.468*1e-9/2*100:.2f} cm')
print(f'10 ns pulse → dead zone = {3e8/1.468*10e-9/2:.3f} m')

## §3 — Fresnel Reflection: Cut vs Tap vs Bend

In [ ]:
section('Reflection and Loss Signatures')

n1, n2 = symbols('n_1 n_2', positive=True)

# Fresnel reflection coefficient (normal incidence)
R_fresnel = ((n1 - n2)/(n1 + n2))**2
step('Fresnel reflectance R = [(n₁−n₂)/(n₁+n₂)]² =', R_fresnel)

# Glass-air interface (cleaved end or break)
R_glass_air = R_fresnel.subs([(n1, 1.468), (n2, 1.0)])
step('Glass/air (cleaved end): R =', float(R_glass_air))
print(f'  = {float(R_glass_air)*100:.2f}%  = {10*float(log(R_glass_air,10)):.1f} dB')

display(Markdown(r'''
**Event classification by OTDR signature**:

```
Power (dB)
│
│\\                          ← normal Rayleigh slope
│  \\____                   ← splice loss (step, no reflection)
│       ▲\\                 ← connector (spike + step)
│          \\\\             ← increased loss section (bend/strain)
│              ▲            ← fiber end (Fresnel spike)
│               ___         ← noise floor
└──────────────────────── distance
```

**Tap signature** (adversarial):
- Small power loss (−0.1 to −1 dB) at tap location
- No reflection (evanescent coupler doesn't create index discontinuity)
- Almost invisible to standard OTDR
- **Detectable with phase-sensitive OTDR** (φ-OTDR): measures phase of backscatter,
  not just amplitude → tap changes local interference pattern
'''))

## §4 — Phase-Sensitive OTDR (φ-OTDR) + TDGSA

In [ ]:
section('φ-OTDR: Phase of Backscatter')

display(Markdown(r'''
**Standard OTDR**: measures $|E_{back}(t)|^2$ — intensity only, phase lost.

**φ-OTDR**: measures the full complex backscattered field $E_{back}(t)$.
Any perturbation (strain, temperature, tap) changes the local phase of backscatter.

**TDGSA extension**:
Instead of a coherent local oscillator (expensive, unstable), use two dispersed
measurements of the backscattered pulse:

```
Probe pulse (broadband, 1550 nm)
        │
        ▼
    Fiber under test (FUT, up to 10,000 km)
        │
        │ backscattered field E_back(t)
        ▼
    Circulator
        │
        ├────────────────────────┐
        │ (direct)               │ (through DCF, β₂z = −5000 ps²)
        ▼                        ▼
    Detector 1               Detector 2
    I_t = |E_back|²          I_d = |D{E_back}|²
        │                        │
        └──────────┬─────────────┘
                   ▼
              TDGSA on RPi CM4
                   │
                   ▼
          Recovered phase φ_back(t)
          → strain/tap/cut map vs distance
```
'''))

# Phase sensitivity to strain
epsilon, L_seg, lam_probe, n_fiber = symbols('epsilon L lambda n', positive=True)
pe = symbols('p_e', positive=True)  # photoelastic coefficient

# Phase change from strain
Delta_phi_strain = 2*pi*n_fiber*L_seg/lam_probe * (1 - pe) * epsilon
step('Phase change from axial strain ε:', Delta_phi_strain)
step('For silica: p_e ≈ 0.22, n=1.468, λ=1550 nm')

# Numeric: 1 microstrain over 1m
import math
delta_phi_num = 2*math.pi*1.468*1.0/1550e-9 * (1-0.22) * 1e-6
print(f'1 με strain over 1 m: Δφ = {delta_phi_num:.4f} rad')
print(f'  Detectable if phase noise < {delta_phi_num:.4f} rad')
print(f'  TDGSA phase accuracy ~ σ_q/|A| ≈ 0.01 rad at 8-bit ADC, good SNR')
print(f'  → sub-microstrain sensitivity achievable')

## §5 — Transpacific Cable: Scale and Geometry

In [ ]:
section('Undersea Cable Infrastructure')

display(Markdown(r'''
**Global undersea fiber network**:
- >400 active cables, ~1.3 million km total
- Carry >95% of intercontinental internet and financial data
- Transpacific (US–Japan): ~9,000–12,000 km
- Transatlantic (US–UK): ~6,000–7,000 km
- Repair time after cut: 2–6 weeks (ship + ROV)
- Cost of transpacific cable: ~$500M

**Known threats**:
| Threat | Mechanism | Detection difficulty |
|---|---|---|
| Anchor drag | mechanical cut | easy (total loss) |
| Fishing trawl | mechanical cut | easy |
| ROV tap | evanescent coupler | **hard** (no reflection) |
| Acoustic monitoring | ship/submarine nearby | separate sensor |
| Repeater compromise | electronics in amplifier hut | supply chain |

**TDGSA ship-borne system monitors for all optical signatures**.
'''))

# Repeater spacing
L_span = symbols('L_span', positive=True)
alpha_dB = 0.2  # dB/km
G_dB = 20.0     # EDFA gain dB
span_km = G_dB / alpha_dB
print(f'Repeater spacing: G/α = {G_dB}/{alpha_dB} = {span_km:.0f} km')
print(f'Transpacific (10,000 km): ~{10000//span_km:.0f} repeaters')

# OTDR range limit
dynamic_range_dB = 45  # typical high-end OTDR
max_range = dynamic_range_dB / (2 * alpha_dB)
print(f'\nOTDR dynamic range {dynamic_range_dB} dB → max range {max_range:.0f} km')
print(f'→ Need ship at cable landing station OR distributed inline monitors')
print(f'→ TDGSA inline monitor: one unit per repeater hut (~{10000//span_km:.0f} units)')

## §6 — Kramers-Kronig: Causality Constrains the Fiber

In [ ]:
section('Causality: Kramers-Kronig Relations')

display(Markdown(r'''
The fiber's complex refractive index $\tilde{n}(\omega) = n_r(\omega) + in_i(\omega)$
must satisfy **causality** (no response before stimulus).

This forces the real and imaginary parts to be Hilbert transform pairs:

$$n_r(\omega) - 1 = \frac{2}{\pi}\mathcal{P}\int_0^\infty \frac{\omega'\, n_i(\omega')}{\omega'^2 - \omega^2}d\omega'$$

$$n_i(\omega) = -\frac{2\omega}{\pi}\mathcal{P}\int_0^\infty \frac{n_r(\omega') - 1}{\omega'^2 - \omega^2}d\omega'$$

**Consequence for TDGSA**:
- $n_i(\omega)$ = absorption spectrum (measurable from loss $\alpha$)
- $n_r(\omega)$ = dispersion (determines $\beta_2$)
- They are **not independent** — knowing one gives the other
- If the fiber is tapped and the tap changes $n_i$ locally,
  $n_r$ must also change → detectable phase shift via KK
'''))

# Sellmeier equation for silica (approximate KK result)
lam_sym = symbols('lambda', positive=True)
# Three-term Sellmeier
B1, B2, B3 = 0.6961663, 0.4079426, 0.8974794
C1, C2, C3 = 0.0684043**2, 0.1162414**2, 9.896161**2  # in microns²

display(Markdown(r'''
**Sellmeier equation** for fused silica (empirical KK result):
$$n^2(\lambda) = 1 + \frac{B_1\lambda^2}{\lambda^2-C_1} + \frac{B_2\lambda^2}{\lambda^2-C_2} + \frac{B_3\lambda^2}{\lambda^2-C_3}$$
'''))

import numpy as np
lam_vals = np.linspace(1.2, 1.7, 100)  # microns
n_sq = 1 + B1*lam_vals**2/(lam_vals**2-C1) + B2*lam_vals**2/(lam_vals**2-C2) + B3*lam_vals**2/(lam_vals**2-C3)
n_vals = np.sqrt(n_sq)
idx_1550 = np.argmin(np.abs(lam_vals - 1.55))
print(f'n(1550 nm) = {n_vals[idx_1550]:.6f}')

# GVD from second derivative of n
# β₂ = (λ³/2πc²) d²n/dλ²
dn = np.gradient(n_vals, lam_vals)
d2n = np.gradient(dn, lam_vals)
c_ms = 3e14  # μm/s
lam_m = lam_vals * 1e-6
beta2 = lam_m**3 / (2*np.pi*c_ms**2) * d2n / (1e-6)**2 * 1e24  # ps²/km
print(f'β₂(1550 nm) ≈ {beta2[idx_1550]:.1f} ps²/km  (measured: −21.7 ps²/km)')
print(f'Zero dispersion wavelength ≈ {lam_vals[np.argmin(np.abs(d2n))]:.3f} μm')

## §7 — Ship-Borne System Architecture

In [ ]:
section('System Architecture')

display(Markdown(r'''
```
SHIP-BORNE TDGSA CABLE MONITOR
════════════════════════════════════════════════════════

  Mode-locked laser (1550 nm, 100 MHz rep rate, 1 ps pulse)
           │
           ▼
      EDFA (booster, +30 dBm launch)
           │
           ▼
      Circulator ──────────────────────► Wet plant connector
           │                              (cable landing or
           │ backscatter                   splice to FUT)
           ▼
      50/50 splitter
      ┌────┴─────────────────┐
      │ (direct)             │ (DCF spool, β₂z = −5000 ps²)
      ▼                      ▼
  InGaAs PD 1           InGaAs PD 2
  1 GSa/s, 12-bit       1 GSa/s, 12-bit
      │                      │
      └──────────┬────────────┘
                 ▼
           RPi CM4 + FPGA
           TDGSA (N=1024, n_iter=50)
           φ(t) → d = ct/2n
                 │
                 ▼
           Alert system
           ├─ Cut: power drop > 10 dB
           ├─ Tap: phase anomaly, no power drop
           ├─ Strain: slope change in α(z)
           └─ Location: ±10 m at 10,000 km
```

**Localization accuracy**:
$$\delta d = \frac{c}{2n}\cdot\delta t = \frac{c}{2n\cdot f_s}$$
At $f_s = 1$ GSa/s: $\delta d \approx 10$ cm per sample.
With pulse compression (matched filter): ±10 m at any range.

**Key advantage over standard OTDR**:
Standard OTDR misses taps (no reflection, small loss).
TDGSA recovers the **phase** of backscatter — a tap changes
the local interference pattern detectably even at −0.1 dB insertion loss.
'''))

f_samp = 1e9
n_f = 1.468
c_v = 3e8
delta_d = c_v / (2*n_f*f_samp)
print(f'Spatial resolution per sample: {delta_d*100:.2f} cm')
print(f'With 1024-point pulse compression: {delta_d*100/32:.3f} cm RMS')

## §8 — DoD / OUSD Alignment and Funding

In [ ]:
section('Strategic alignment')

display(Markdown(r'''
**Why this matters to DoD** (not classified, all open-source reasoning):

| Threat | DoD concern | This system addresses |
|---|---|---|
| Undersea cable cut | Communications blackout | Immediate localization |
| Cable tap | Intelligence collection | Phase-sensitive tap detection |
| Repeater hardware compromise | Supply chain | Anomalous phase signature |
| Adversarial survey ship | Pre-positioning | Distributed monitoring record |

**Relevant DoD programs**:
- **DARPA SUBS** (Submarine systems) — undersea domain awareness
- **NAVSEA PMS 395** — undersea communications infrastructure
- **CISA** — critical infrastructure / cable protection
- **NSF PAWR** — platforms for advanced wireless research (dual-use fiber)
- **SBIR Topic N24-089** (Navy) — fiber sensing for undersea systems

**Civilian funding path** (non-DoD, faster):
- **Telecom operators** (AT&T, Lumen, SubCom): pay directly for cable monitoring
- **Lloyd's / maritime insurance**: reduce claim exposure from cable damage
- **ITU**: international cable protection frameworks

**What you need to pitch**:
1. Demo: run TDGSA on a fiber spool with a simulated tap (bend coupler)
2. Show: phase anomaly detected at < −0.5 dB insertion loss
3. Compare: standard OTDR misses it, TDGSA catches it
4. One-pager: cost ($50K/unit vs $2B/repair ship) + localization accuracy

**Jalali lab connection**: this is a direct extension of the STEAM camera
and TS-DFT work. Same hardware, different application. One paper:
"Phase-sensitive OTDR using dispersive Fourier transform and
Gerchberg-Saxton phase retrieval" — publishable in Optics Letters or
Journal of Lightwave Technology.
'''))